In [1]:
import sqlite3
import requests

# 1. الاتصال بقاعدة البيانات وإنشاء الجدول (نفس قاعدة بيانات store.db)
conn = sqlite3.connect('store.db')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS products (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        category TEXT NOT NULL,
        price REAL NOT NULL
    )
''')
cursor.execute('DELETE FROM products') # مسح البيانات القديمة لضمان نظافة الجدول
conn.commit()

# 2. إرسال GET request للـ API المجاني
url = "https://jsonplaceholder.typicode.com/posts"  # API تجريبي عام
response = requests.get(url)

# 3. التحقق من نجاح الطلب (Status Code 200)
if response.status_code == 200:
    print("--- تم الاتصال بالـ API بنجاح (Status Code: 200) ---")
    data = response.json()

    # 4. استخراج البيانات (ناخد أول 5 عناصر كمثال ونكيفها على أعمدة الجدول: name, category, price)
    # بناءً على الـ API ده، هنعتبر title هو الـ name، و body هو الـ category، و id * 100 هو الـ price
    extracted_products = []
    for item in data[:5]:
        name = item['title'][:15]         # أخذ أول 15 حرف للاسم
        category = "General"              # فئة ثابتة للتجربة
        price = float(item['id'] * 50)    # سعر تقريبي بناءً على الـ id
        extracted_products.append((name, category, price))

    # 5. إدراج البيانات المستخرجة في قاعدة البيانات store.db
    cursor.executemany('''
        INSERT INTO products (name, category, price)
        VALUES (?, ?, ?)
    ''', extracted_products)
    conn.commit()
    print("--- تم تخزين بيانات الـ API في قاعدة البيانات بنجاح ---\n")

else:
    print(f"فشل الاتصال، كود الخطأ: {response.status_code}")

# 6. التنفيذ والاستعلام الأول: استخدام WHERE لاسترجاع فئة معينة
print("=== الاستعلام الأول: المنتجات التابعة لفئة General (WHERE) ===")
cursor.execute("SELECT * FROM products WHERE category = 'General'")
for row in cursor.fetchall():
    print(row)

print("\n")

# 7. التنفيذ والاستعلام الثاني: استخدام ORDER BY price DESC لترتيب المنتجات من الأعلى للأقل سعراً
print("=== الاستعلام الثاني: المنتجات مرتبة تنازلياً حسب السعر (ORDER BY price DESC) ===")
cursor.execute("SELECT * FROM products ORDER BY price DESC")
for row in cursor.fetchall():
    print(row)

# إغلاق الاتصال بقاعدة البيانات
conn.close()

--- تم الاتصال بالـ API بنجاح (Status Code: 200) ---
--- تم تخزين بيانات الـ API في قاعدة البيانات بنجاح ---

=== الاستعلام الأول: المنتجات التابعة لفئة General (WHERE) ===
(1, 'sunt aut facere', 'General', 50.0)
(2, 'qui est esse', 'General', 100.0)
(3, 'ea molestias qu', 'General', 150.0)
(4, 'eum et est occa', 'General', 200.0)
(5, 'nesciunt quas o', 'General', 250.0)


=== الاستعلام الثاني: المنتجات مرتبة تنازلياً حسب السعر (ORDER BY price DESC) ===
(5, 'nesciunt quas o', 'General', 250.0)
(4, 'eum et est occa', 'General', 200.0)
(3, 'ea molestias qu', 'General', 150.0)
(2, 'qui est esse', 'General', 100.0)
(1, 'sunt aut facere', 'General', 50.0)
